In [8]:
!pip install pulp

In [10]:
import pulp

# ==========================================
# 1. SETS & INDICES
# ==========================================
MILLS = ['Mill_China', 'Mill_India']
FACTORIES = ['Fact_Vietnam', 'Fact_Bangladesh', 'Fact_Mexico']
DCS = ['DC_US_East', 'DC_US_West']
MARKETS = ['Market_NY', 'Market_LA', 'Market_Chicago']

# ==========================================
# 2. PARAMETERS
# ==========================================
# Production / Assembly Costs
PC_mill = {'Mill_China': 2.50, 'Mill_India': 2.20}          # $ per meter of fabric
PC_factory = {'Fact_Vietnam': 4.00, 'Fact_Bangladesh': 3.50, 'Fact_Mexico': 5.00} # $ per garment

# Capacities
Cap_mill = {'Mill_China': 500000, 'Mill_India': 400000}     # meters of fabric
Cap_factory = {'Fact_Vietnam': 200000, 'Fact_Bangladesh': 250000, 'Fact_Mexico': 150000} # garments
Cap_dc = {'DC_US_East': 250000, 'DC_US_West': 250000}       # garments

# Customer Demand
Dem_market = {'Market_NY': 150000, 'Market_LA': 120000, 'Market_Chicago': 80000} # garments

# Yield & Conversion Parameters
gamma = 1.5  # 1.5 meters required per garment
alpha = {'Fact_Vietnam': 0.96, 'Fact_Bangladesh': 0.92, 'Fact_Mexico': 0.98} # Factory efficiency/yield

# Transportation Costs (Nested Dicts)
# TC_mf: Mill to Factory ($ per meter)
TC_mf = {
    'Mill_China':    {'Fact_Vietnam': 0.10, 'Fact_Bangladesh': 0.15, 'Fact_Mexico': 0.35},
    'Mill_India':    {'Fact_Vietnam': 0.18, 'Fact_Bangladesh': 0.08, 'Fact_Mexico': 0.38}
}

# TC_fd: Factory to DC ($ per garment)
TC_fd = {
    'Fact_Vietnam':    {'DC_US_East': 0.85, 'DC_US_West': 0.60},
    'Fact_Bangladesh': {'DC_US_East': 0.90, 'DC_US_West': 0.70},
    'Fact_Mexico':     {'DC_US_East': 0.40, 'DC_US_West': 0.45}
}

# TC_dj: DC to Market ($ per garment)
TC_dj = {
    'DC_US_East': {'Market_NY': 0.15, 'Market_LA': 0.50, 'Market_Chicago': 0.30},
    'DC_US_West': {'Market_NY': 0.48, 'Market_LA': 0.12, 'Market_Chicago': 0.28}
}

In [11]:
# Initialize the Optimization Problem
model = pulp.LpProblem("Apparel_Supply_Chain_Optimization", pulp.LpMinimize)

# Decision Variables
X = pulp.LpVariable.dicts("Fabric_Flow_mf", ((m, f) for m in MILLS for f in FACTORIES), lowBound=0, cat='Continuous')
Y = pulp.LpVariable.dicts("Garment_Flow_fd", ((f, d) for f in FACTORIES for d in DCS), lowBound=0, cat='Continuous')
Z = pulp.LpVariable.dicts("Garment_Flow_dj", ((d, j) for d in DCS for j in MARKETS), lowBound=0, cat='Continuous')

In [12]:
# ==========================================
# 3. OBJECTIVE FUNCTION
# ==========================================
model += (
    pulp.lpSum((PC_mill[m] + TC_mf[m][f]) * X[m, f] for m in MILLS for f in FACTORIES) +
    pulp.lpSum((PC_factory[f] + TC_fd[f][d]) * Y[f, d] for f in FACTORIES for d in DCS) +
    pulp.lpSum(TC_dj[d][j] * Z[d, j] for d in DCS for j in MARKETS)
), "Total_Supply_Chain_Cost"

# ==========================================
# 4. CONSTRAINTS
# ==========================================

# 4.1 Capacity Constraints
for m in MILLS:
    model += pulp.lpSum(X[m, f] for f in FACTORIES) <= Cap_mill[m], f"Mill_Cap_{m}"

for f in FACTORIES:
    model += pulp.lpSum(Y[f, d] for d in DCS) <= Cap_factory[f], f"Factory_Cap_{f}"

for d in DCS:
    model += pulp.lpSum(Z[d, j] for j in MARKETS) <= Cap_dc[d], f"DC_Cap_{d}"

# 4.2 Demand Constraint
for j in MARKETS:
    model += pulp.lpSum(Z[d, j] for d in DCS) >= Dem_market[j], f"Demand_Sat_{j}"

# 4.3 Flow Conservation Constraints
for d in DCS:
    model += pulp.lpSum(Y[f, d] for f in FACTORIES) == pulp.lpSum(Z[d, j] for j in MARKETS), f"DC_Flow_Balance_{d}"

# 4.4 Factory BOM & Yield Material Balance (The critical industry constraint!)
for f in FACTORIES:
    model += (
        pulp.lpSum(X[m, f] * alpha[f] for m in MILLS) == gamma * pulp.lpSum(Y[f, d] for d in DCS),
        f"Factory_BOM_Balance_{f}"
    )

In [15]:
# Solve the model
status = model.solve()

# Print the solution status
print(f"Optimization Status: {pulp.LpStatus[status]}")
print(f"Minimized Total Cost: ${model.objective.value():,.2f}\n")

print("="*40)
print("OPTIMAL NETWORK FLOWS")
print("="*40)

print("\n--- 1. Fabric Shipped from Mills to Factories (Meters) ---")
for m in MILLS:
    for f in FACTORIES:
        val = X[m, f].varValue
        if val and val > 0:
            print(f"  {m} ──> {f}: {val:,.2f} meters")

print("\n--- 2. Garments Shipped from Factories to DCs (Units) ---")
for f in FACTORIES:
    for d in DCS:
        val = Y[f, d].varValue
        if val and val > 0:
            print(f"  {f} ──> {d}: {val:,.2f} units")

print("\n--- 3. Garments Shipped from DCs to Markets (Units) ---")
for d in DCS:
    for j in MARKETS:
        val = Z[d, j].varValue
        if val and val > 0:
            print(f"  {d} ──> {j}: {val:,.2f} units")

Optimization Status: Optimal
Minimized Total Cost: $2,937,713.04

OPTIMAL NETWORK FLOWS

--- 1. Fabric Shipped from Mills to Factories (Meters) ---
  Mill_China ──> Fact_Vietnam: 156,250.00 meters
  Mill_China ──> Fact_Bangladesh: 7,608.70 meters
  Mill_India ──> Fact_Bangladesh: 400,000.00 meters

--- 2. Garments Shipped from Factories to DCs (Units) ---
  Fact_Vietnam ──> DC_US_West: 100,000.00 units
  Fact_Bangladesh ──> DC_US_East: 150,000.00 units
  Fact_Bangladesh ──> DC_US_West: 100,000.00 units

--- 3. Garments Shipped from DCs to Markets (Units) ---
  DC_US_East ──> Market_NY: 150,000.00 units
  DC_US_West ──> Market_LA: 120,000.00 units
  DC_US_West ──> Market_Chicago: 80,000.00 units


In [16]:
print("="*40)
print("SENIOR IE PERFORMANCE METRICS")
print("="*40)

print("\n[Factory Capacity Utilization]")
for f in FACTORIES:
    # Total garments produced by factory f
    total_produced = sum(Y[f, d].varValue for d in DCS if Y[f, d].varValue)
    utilization = (total_produced / Cap_factory[f]) * 100
    print(f"  {f}: {total_produced:,.0f} / {Cap_factory[f]:,.0f} units ({utilization:.1f}% utilized)")

print("\n[Fabric Wastage Analysis]")
for f in FACTORIES:
    # Total fabric entering the factory
    fabric_in = sum(X[m, f].varValue for m in MILLS if X[m, f].varValue)
    # Wasted fabric due to inefficiency (1 - alpha)
    fabric_wasted = fabric_in * (1 - alpha[f])

    if fabric_in > 0:
        print(f"  {f}: Shipped in {fabric_in:,.0f}m | Scrapped {fabric_wasted:,.0f}m due to {((1-alpha[f])*100):.0f}% cutting waste")

SENIOR IE PERFORMANCE METRICS

[Factory Capacity Utilization]
  Fact_Vietnam: 100,000 / 200,000 units (50.0% utilized)
  Fact_Bangladesh: 250,000 / 250,000 units (100.0% utilized)
  Fact_Mexico: 0 / 150,000 units (0.0% utilized)

[Fabric Wastage Analysis]
  Fact_Vietnam: Shipped in 156,250m | Scrapped 6,250m due to 4% cutting waste
  Fact_Bangladesh: Shipped in 407,609m | Scrapped 32,609m due to 8% cutting waste


In [17]:
import copy

print("="*50)
print("RUNNING SCENARIO: 50% ASIA OCEAN FREIGHT SPIKE")
print("="*50)

# 1. Clone our original factory-to-DC shipping costs so we don't overwrite baseline data
TC_fd_disrupted = copy.deepcopy(TC_fd)

# 2. Apply a 50% cost increase to Asian factories (Vietnam and Bangladesh)
for f in ['Fact_Vietnam', 'Fact_Bangladesh']:
    for d in DCS:
        TC_fd_disrupted[f][d] *= 1.50
        print(f"  New Freight Cost {f} ──> {d}: ${TC_fd_disrupted[f][d]:.2f}")

# 3. Re-initialize a new optimization problem
model_scenario = pulp.LpProblem("Apparel_Supply_Chain_Disruption", pulp.LpMinimize)

# Decision Variables
X_s = pulp.LpVariable.dicts("Fabric_Flow_mf", ((m, f) for m in MILLS for f in FACTORIES), lowBound=0, cat='Continuous')
Y_s = pulp.LpVariable.dicts("Garment_Flow_fd", ((f, d) for f in FACTORIES for d in DCS), lowBound=0, cat='Continuous')
Z_s = pulp.LpVariable.dicts("Garment_Flow_dj", ((d, j) for d in DCS for j in MARKETS), lowBound=0, cat='Continuous')

# Objective Function (Using the new disrupted shipping costs!)
model_scenario += (
    pulp.lpSum((PC_mill[m] + TC_mf[m][f]) * X_s[m, f] for m in MILLS for f in FACTORIES) +
    pulp.lpSum((PC_factory[f] + TC_fd_disrupted[f][d]) * Y_s[f, d] for f in FACTORIES for d in DCS) +
    pulp.lpSum(TC_dj[d][j] * Z_s[d, j] for d in DCS for j in MARKETS)
), "Total_Disrupted_Supply_Chain_Cost"

# Constraints (Same logic as before)
for m in MILLS: model_scenario += pulp.lpSum(X_s[m, f] for f in FACTORIES) <= Cap_mill[m]
for f in FACTORIES: model_scenario += pulp.lpSum(Y_s[f, d] for d in DCS) <= Cap_factory[f]
for d in DCS: model_scenario += pulp.lpSum(Z_s[d, j] for j in MARKETS) <= Cap_dc[d]
for j in MARKETS: model_scenario += pulp.lpSum(Z_s[d, j] for d in DCS) >= Dem_market[j]
for d in DCS: model_scenario += pulp.lpSum(Y_s[f, d] for f in FACTORIES) == pulp.lpSum(Z_s[d, j] for j in MARKETS)
for f in FACTORIES:
    model_scenario += pulp.lpSum(X_s[m, f] * alpha[f] for m in MILLS) == gamma * pulp.lpSum(Y_s[f, d] for d in DCS)

# Solve the disrupted model
model_scenario.solve()

print(f"\nNew Disrupted Total Cost: ${model_scenario.objective.value():,.2f}")
print(f"Cost Increase vs Baseline: ${model_scenario.objective.value() - model.objective.value():,.2f}\n")

print("[New Factory Capacity Utilization]")
for f in FACTORIES:
    total_produced = sum(Y_s[f, d].varValue for d in DCS if Y_s[f, d].varValue)
    utilization = (total_produced / Cap_factory[f]) * 100
    print(f"  {f}: {total_produced:,.0f} / {Cap_factory[f]:,.0f} units ({utilization:.1f}% utilized)")

RUNNING SCENARIO: 50% ASIA OCEAN FREIGHT SPIKE
  New Freight Cost Fact_Vietnam ──> DC_US_East: $1.27
  New Freight Cost Fact_Vietnam ──> DC_US_West: $0.90
  New Freight Cost Fact_Bangladesh ──> DC_US_East: $1.35
  New Freight Cost Fact_Bangladesh ──> DC_US_West: $1.05

New Disrupted Total Cost: $3,070,213.04
Cost Increase vs Baseline: $132,500.00

[New Factory Capacity Utilization]
  Fact_Vietnam: 100,000 / 200,000 units (50.0% utilized)
  Fact_Bangladesh: 250,000 / 250,000 units (100.0% utilized)
  Fact_Mexico: 0 / 150,000 units (0.0% utilized)


In [18]:
import pandas as pd

# 1. Re-run Baseline to ensure fresh variables
model_base = pulp.LpProblem("Baseline", pulp.LpMinimize)
X_b = pulp.LpVariable.dicts("X_b", ((m, f) for m in MILLS for f in FACTORIES), lowBound=0)
Y_b = pulp.LpVariable.dicts("Y_b", ((f, d) for f in FACTORIES for d in DCS), lowBound=0)
Z_b = pulp.LpVariable.dicts("Z_b", ((d, j) for d in DCS for j in MARKETS), lowBound=0)

model_base += (pulp.lpSum((PC_mill[m] + TC_mf[m][f]) * X_b[m, f] for m in MILLS for f in FACTORIES) +
               pulp.lpSum((PC_factory[f] + TC_fd[f][d]) * Y_b[f, d] for f in FACTORIES for d in DCS) +
               pulp.lpSum(TC_dj[d][j] * Z_b[d, j] for d in DCS for j in MARKETS))

for m in MILLS: model_base += pulp.lpSum(X_b[m, f] for f in FACTORIES) <= Cap_mill[m]
for f in FACTORIES: model_base += pulp.lpSum(Y_b[f, d] for d in DCS) <= Cap_factory[f]
for d in DCS: model_base += pulp.lpSum(Z_b[d, j] for j in MARKETS) <= Cap_dc[d]
for j in MARKETS: model_base += pulp.lpSum(Z_b[d, j] for d in DCS) >= Dem_market[j]
for d in DCS: model_base += pulp.lpSum(Y_b[f, d] for f in FACTORIES) == pulp.lpSum(Z_b[d, j] for j in MARKETS)
for f in FACTORIES: model_base += pulp.lpSum(X_b[m, f] * alpha[f] for m in MILLS) == gamma * pulp.lpSum(Y_b[f, d] for d in DCS)
model_base.solve()

# 2. Setup CORRECT Disrupted Costs (Explicitly copy ALL factories first)
TC_fd_correct = {f: {d: TC_fd[f][d] for d in DCS} for f in FACTORIES}
# Now apply the 50% spike ONLY to Asia
for f in ['Fact_Vietnam', 'Fact_Bangladesh']:
    for d in DCS:
        TC_fd_correct[f][d] *= 1.50

model_disrupt = pulp.LpProblem("Disrupted", pulp.LpMinimize)
X_d = pulp.LpVariable.dicts("X_d", ((m, f) for m in MILLS for f in FACTORIES), lowBound=0)
Y_d = pulp.LpVariable.dicts("Y_d", ((f, d) for f in FACTORIES for d in DCS), lowBound=0)
Z_d = pulp.LpVariable.dicts("Z_d", ((d, j) for d in DCS for j in MARKETS), lowBound=0)

model_disrupt += (pulp.lpSum((PC_mill[m] + TC_mf[m][f]) * X_d[m, f] for m in MILLS for f in FACTORIES) +
                  pulp.lpSum((PC_factory[f] + TC_fd_correct[f][d]) * Y_d[f, d] for f in FACTORIES for d in DCS) +
                  pulp.lpSum(TC_dj[d][j] * Z_d[d, j] for d in DCS for j in MARKETS))

for m in MILLS: model_disrupt += pulp.lpSum(X_d[m, f] for f in FACTORIES) <= Cap_mill[m]
for f in FACTORIES: model_disrupt += pulp.lpSum(Y_d[f, d] for d in DCS) <= Cap_factory[f]
for d in DCS: model_disrupt += pulp.lpSum(Z_d[d, j] for d in DCS) <= Cap_dc[d]
for j in MARKETS: model_disrupt += pulp.lpSum(Z_d[d, j] for d in DCS) >= Dem_market[j]
for d in DCS: model_disrupt += pulp.lpSum(Y_d[f, d] for f in FACTORIES) == pulp.lpSum(Z_d[d, j] for j in MARKETS)
for f in FACTORIES: model_disrupt += pulp.lpSum(X_d[m, f] * alpha[f] for m in MILLS) == gamma * pulp.lpSum(Y_d[f, d] for d in DCS)
model_disrupt.solve()

# 3. Create Pandas Comparison Table
summary_data = []
for f in FACTORIES:
    prod_base = sum(Y_b[f, d].varValue for d in DCS if Y_b[f, d].varValue)
    prod_disrupt = sum(Y_d[f, d].varValue for d in DCS if Y_d[f, d].varValue)
    summary_data.append({
        'Factory': f,
        'Baseline Production': f"{prod_base:,.0f}",
        'Disrupted Production': f"{prod_disrupt:,.0f}",
        'Shift': f"{(prod_disrupt - prod_base):+,.0f}"
    })

df = pd.DataFrame(summary_data)
print("=== STRATEGIC SCENARIO ANALYSIS SUMMARY ===")
print(f"Baseline Total Cost:  ${model_base.objective.value():,.2f}")
print(f"Disrupted Total Cost: ${model_disrupt.objective.value():,.2f}")
print(f"Financial Impact:     +${(model_disrupt.objective.value() - model_base.objective.value()):,.2f}\n")
print(df.to_string(index=False))

=== STRATEGIC SCENARIO ANALYSIS SUMMARY ===
Baseline Total Cost:  $2,937,713.04
Disrupted Total Cost: $3,070,213.04
Financial Impact:     +$132,500.00

        Factory Baseline Production Disrupted Production Shift
   Fact_Vietnam             100,000              100,000    +0
Fact_Bangladesh             250,000              250,000    +0
    Fact_Mexico                   0                    0    +0
